# Module 14 - Preference tuning (DPO)

Use this notebook after `tests/test_dpo.py` is passing and after you have saved a Module 13 `*-SFT` model artifact. The notebook loads the strongest available SFT artifact by default, builds a small preference dataset, checks the step-0 `log(2)` invariant, trains with DPO, compares SFT vs DPO behavior, and saves a DPO artifact for Module 15.

DPO should feel like SFT with one extra dimension: every prompt has a chosen answer and a rejected answer, and the frozen reference model anchors how far the policy is allowed to move.

## Setup

In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import torch

from g2c.artifacts import (
    available_model_artifacts_with_suffix,
    load_model_artifact_with_tokenizer,
    save_huggingface_model_artifact,
    save_model_artifact,
)
from g2c.dpo import (
    DPOTrainer,
    PreferenceExample,
    dpo_loss,
    pad_and_collate_pref,
    sequence_logprob,
)
from g2c.notebook_extras.dpo import plot_dpo_history, train_dpo_with_progress
from g2c.notebook_extras.model_selection import select_sft_artifact_name
from g2c.sampling import generate
from g2c.sft import ChatTemplate

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the DPO tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 14 TODOs in `g2c/dpo/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_dpo.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 14 DPO tests are not passing yet."

## Model selection

BaseLM-SFT is the default because DPO needs a model that already has useful instruction-following behavior. To run DPO on your own model, set `MODEL_SELECTION = "course"` for the strongest course-trained `*-SFT` artifact, or set it to a concrete artifact such as `"StoryLM-30M-SFT"`.

In [ ]:
MODEL_SELECTION = "BaseLM"  # "BaseLM", "course", or an SFT artifact name such as "TinyLLM-30M-SFT"
TRAIN_DEVICE = "auto"
SEED = 14

SFT_ARTIFACT_NAME = select_sft_artifact_name(MODEL_SELECTION, repo_root=repo_root)
print("selected SFT artifact:", SFT_ARTIFACT_NAME)

## Load the selected SFT artifact

DPO holds two model copies at once: a trainable policy and a frozen reference. If memory is tight, choose a smaller artifact or set `TRAIN_DEVICE = "cpu"` for the first debugging pass.

In [ ]:
available_sft = available_model_artifacts_with_suffix("-SFT", repo_root=repo_root)
if available_sft:
    print("available SFT artifacts:")
    for artifact in available_sft:
        print(f"  rank {artifact.rank:>3}: {artifact.name}")
else:
    print("No SFT artifacts found under artifacts/models/.")

policy_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)
reference_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)

policy_model = policy_artifact.model
ref_model = reference_artifact.model
tokenizer = policy_artifact.tokenizer
template = ChatTemplate()
pad_id = tokenizer.special_to_id.get("<|pad|>", getattr(tokenizer, "pad_token_id", None) or 0)
end_id = tokenizer.special_to_id.get(template.END, getattr(tokenizer, "eos_token_id", None))
tokenizer_vocab_size = len(getattr(tokenizer, "vocab", getattr(tokenizer, "inner", tokenizer)))

def model_device(model) -> torch.device:
    device = getattr(model, "device", None)
    if isinstance(device, torch.device):
        return device
    for parameter in model.parameters():
        return parameter.device
    return torch.device("cpu")


print("loaded SFT artifact:", policy_artifact.name)
print("display:", policy_artifact.display_name)
print("kind:", policy_artifact.manifest.get("kind", "course_transformer"))
print("model vocab:", policy_model.vocab_size)
print("tokenizer vocab:", tokenizer_vocab_size)
print("max seq len:", policy_model.max_seq_len)
print("pad id:", pad_id, "end id:", end_id)
print("policy device:", model_device(policy_model))

## Sampling helpers

These are notebook helpers, not the Module 14 deliverable. They render the same chat template used in SFT and stop generation at `<|end|>` when that token exists.

In [ ]:
def chat_prompt(user_text: str, assistant_prefix: str = "") -> str:
    return (
        template.render([{"role": "user", "content": user_text}])
        + f"{template.ASSISTANT}\n"
        + assistant_prefix
    )


def encode_for_model(model, text: str) -> torch.Tensor:
    ids = tokenizer.encode_with_vocab_size(text, model.vocab_size)
    if not ids:
        raise ValueError("prompt encoded to no tokens")
    return torch.tensor(ids, dtype=torch.long)


def sample_from_model(
    model,
    prompt: str,
    *,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float | None = 0.9,
    seed: int = SEED,
) -> str:
    prompt_ids = encode_for_model(model, prompt)
    ids = generate(
        model,
        prompt_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.1,
        eos_id=end_id,
        generator=torch.Generator().manual_seed(seed),
    )
    return tokenizer.decode([int(x) for x in ids.tolist()])


def sample_response(model, user_text: str, **kwargs) -> str:
    return sample_from_model(model, chat_prompt(user_text), **kwargs)


def printable(text: str) -> str:
    has_control = any(ord(ch) < 32 and ch not in "\n\t" for ch in text)
    return text.encode("unicode_escape").decode("ascii") if has_control else text


def show_response(label: str, model, user_text: str, **kwargs) -> str:
    text = sample_response(model, user_text, **kwargs)
    print("-" * 72)
    print(label)
    print(printable(text))
    return text

A quick baseline read on the SFT model. This is not graded; it gives you a qualitative anchor before DPO changes the policy.

In [ ]:
BASELINE_PROMPTS = [
    "What is the capital of Spain?",
    "Answer in one short sentence: what is a neural network?",
    "If you are unsure, how should you answer?",
]

for prompt in BASELINE_PROMPTS:
    print("=" * 72)
    print("user:", prompt)
    show_response("SFT model", ref_model, prompt, max_new_tokens=80, seed=SEED)

## Exercise 1 - Build preference pairs

The starter set below is large enough to make validation less jumpy while still staying small enough for local experiments. It covers several preference axes: factual correctness, arithmetic, format following, honesty, course help, debugging advice, concise explanation style, grounding, tool-call JSON, and answering the actual question.

Each row should vary one axis at a time. Avoid pairing a long polished chosen answer with a short broken rejected answer unless the preference you want to teach is length.

In [ ]:
preference_rows = [
    {"kind": "factual", "user": "What is the capital of France?", "chosen": "Paris.", "rejected": "London."},
    {"kind": "factual", "user": "What is the capital of Spain?", "chosen": "Madrid.", "rejected": "Lisbon."},
    {"kind": "factual", "user": "What is the capital of Italy?", "chosen": "Rome.", "rejected": "Milan."},
    {"kind": "factual", "user": "What is the capital of Germany?", "chosen": "Berlin.", "rejected": "Munich."},
    {"kind": "factual", "user": "What is the capital of Japan?", "chosen": "Tokyo.", "rejected": "Kyoto."},
    {"kind": "factual", "user": "Which planet is known as the red planet?", "chosen": "Mars.", "rejected": "Venus."},
    {"kind": "factual", "user": "What gas do plants take in for photosynthesis?", "chosen": "Carbon dioxide.", "rejected": "Oxygen."},
    {"kind": "factual", "user": "What do bees make?", "chosen": "Honey.", "rejected": "Milk."},
    {"kind": "arithmetic", "user": "What is 2 + 3?", "chosen": "5.", "rejected": "6."},
    {"kind": "arithmetic", "user": "What is 4 * 5?", "chosen": "20.", "rejected": "9."},
    {"kind": "arithmetic", "user": "What is 12 - 7?", "chosen": "5.", "rejected": "19."},
    {"kind": "arithmetic", "user": "What is 8 + 6?", "chosen": "14.", "rejected": "13."},
    {"kind": "arithmetic", "user": "What is 9 / 3?", "chosen": "3.", "rejected": "6."},
    {"kind": "arithmetic", "user": "What is 6 + 7?", "chosen": "13.", "rejected": "12."},
    {"kind": "arithmetic", "user": "What is 15 - 6?", "chosen": "9.", "rejected": "21."},
    {"kind": "arithmetic", "user": "What is 3 * 8?", "chosen": "24.", "rejected": "11."},
    {"kind": "format", "user": "Return exactly one color.", "chosen": "Blue.", "rejected": "Blue is a color that many people like."},
    {"kind": "format", "user": "Return exactly one number: seven.", "chosen": "7.", "rejected": "The number is 7, which is seven."},
    {"kind": "format", "user": "Answer yes or no: is water wet?", "chosen": "Yes.", "rejected": "Water can be considered wet in many contexts."},
    {"kind": "format", "user": "Answer with one word: cat or dog?", "chosen": "Cat.", "rejected": "I would choose cat because cats are nice."},
    {"kind": "format", "user": "Return lowercase only: HELLO.", "chosen": "hello", "rejected": "Hello is now lowercase: hello."},
    {"kind": "format", "user": "Return uppercase only: quiet.", "chosen": "QUIET", "rejected": "The uppercase version is QUIET."},
    {"kind": "format", "user": "Return JSON with color blue.", "chosen": "{\"color\":\"blue\"}", "rejected": "The JSON is {\"color\":\"blue\"}."},
    {"kind": "format", "user": "Return exactly two bullet items: red and blue.", "chosen": "- red\n- blue", "rejected": "Here are two colors:\n- red\n- blue\nThey are common colors."},
    {"kind": "format", "user": "Answer with one word: opposite of hot.", "chosen": "Cold.", "rejected": "The opposite of hot is cold."},
    {"kind": "format", "user": "Return only the filename: report.md", "chosen": "report.md", "rejected": "The filename is report.md."},
    {"kind": "honesty", "user": "If you are not sure about an answer, what should you say?", "chosen": "I am not sure.", "rejected": "I know for certain."},
    {"kind": "honesty", "user": "If a fact might be outdated, what should an assistant do?", "chosen": "Say it may need verification.", "rejected": "State it confidently anyway."},
    {"kind": "honesty", "user": "Should an assistant invent a source when it lacks one?", "chosen": "No, it should say it does not have a source.", "rejected": "Yes, it should make one sound plausible."},
    {"kind": "honesty", "user": "If the prompt is ambiguous, what is a good response?", "chosen": "Ask a clarifying question or state the assumption.", "rejected": "Guess silently and pretend it was clear."},
    {"kind": "honesty", "user": "If you cannot see a file, what should you say?", "chosen": "I cannot see that file unless you provide it.", "rejected": "I checked the file and it looks correct."},
    {"kind": "honesty", "user": "If there is not enough information, what should you do?", "chosen": "Say what information is missing.", "rejected": "Fill in the missing details confidently."},
    {"kind": "honesty", "user": "Should an assistant claim a test passed if it did not run it?", "chosen": "No, it should say the test was not run.", "rejected": "Yes, it can say the test passed anyway."},
    {"kind": "honesty", "user": "If an answer depends on local files you have not read, what should you do?", "chosen": "Inspect the files or state the uncertainty.", "rejected": "Assume the file contents from memory."},
    {"kind": "honesty", "user": "If two instructions conflict, what should an assistant do?", "chosen": "Follow the higher-priority or newer instruction and note the conflict if needed.", "rejected": "Ignore the conflict and follow both."},
    {"kind": "honesty", "user": "If a command fails, what should the summary say?", "chosen": "Report that it failed and include the relevant error.", "rejected": "Say the command succeeded to keep the summary short."},
    {"kind": "course_help", "user": "Why do we use gradients in this course?", "chosen": "Gradients show how a loss should nudge each parameter to improve predictions.", "rejected": "Gradients are just a math detail and do not affect learning."},
    {"kind": "course_help", "user": "What does a tensor shape tell you?", "chosen": "It tells you how many axes the data has and how large each axis is.", "rejected": "It tells you the exact values stored in the tensor."},
    {"kind": "course_help", "user": "Why does BPE merge frequent byte pairs?", "chosen": "Frequent merges create reusable tokens for common text patterns.", "rejected": "BPE merges random bytes so the vocabulary looks varied."},
    {"kind": "course_help", "user": "What does softmax do to logits?", "chosen": "Softmax turns logits into probabilities that sum to one.", "rejected": "Softmax sorts logits from largest to smallest."},
    {"kind": "course_help", "user": "What does validation loss help detect?", "chosen": "It helps detect whether improvements generalize beyond the training data.", "rejected": "It proves the model has memorized the training examples."},
    {"kind": "course_help", "user": "Why use AdamW instead of plain SGD for transformers?", "chosen": "AdamW adapts update scale per parameter and decouples weight decay.", "rejected": "AdamW is the same as SGD but with a different name."},
    {"kind": "course_help", "user": "What does DPO compare?", "chosen": "DPO compares chosen and rejected responses relative to a frozen reference model.", "rejected": "DPO compares two tokenizers and keeps the faster one."},
    {"kind": "course_help", "user": "Why mask user tokens during SFT?", "chosen": "The loss should train the assistant response, not the already-provided user prompt.", "rejected": "Masking user tokens makes the model forget the question."},
    {"kind": "course_help", "user": "What does a causal mask prevent?", "chosen": "It prevents each position from attending to future tokens.", "rejected": "It prevents the model from attending to earlier tokens."},
    {"kind": "course_help", "user": "What is a KV cache for?", "chosen": "It reuses past keys and values so inference does less repeated attention work.", "rejected": "It stores the model weights in a smaller file."},
    {"kind": "course_help", "user": "Why use retrieval in an assistant?", "chosen": "Retrieval adds relevant external context before the model answers.", "rejected": "Retrieval changes the model weights during the prompt."},
    {"kind": "course_help", "user": "What is the role of a tokenizer artifact?", "chosen": "It makes future model runs use the same text-to-token mapping.", "rejected": "It stores the trained neural network weights."},
    {"kind": "debugging", "user": "Give one tip for debugging a failing test.", "chosen": "Run the smallest failing test and inspect the first wrong value.", "rejected": "Tests are annoying, so just change the code until it works."},
    {"kind": "debugging", "user": "What should I do if my training loss is NaN?", "chosen": "Lower the learning rate and check for unstable operations or invalid data.", "rejected": "Keep training; NaN usually fixes itself."},
    {"kind": "debugging", "user": "How do I check whether a model is overfitting?", "chosen": "Compare train loss against validation loss over time.", "rejected": "Only look at the final training loss."},
    {"kind": "debugging", "user": "What should I do before changing many hyperparameters?", "chosen": "Change one thing at a time and record the result.", "rejected": "Change everything at once so the run is different."},
    {"kind": "debugging", "user": "What should I check after a tensor shape error?", "chosen": "Print the relevant shapes and match them to the expected contract.", "rejected": "Add random reshapes until the error disappears."},
    {"kind": "debugging", "user": "What should I do if MPS is unavailable?", "chosen": "Confirm PyTorch sees MPS and fall back to CPU if needed.", "rejected": "Assume the GPU is working because the Mac has Apple Silicon."},
    {"kind": "debugging", "user": "What should I do if a checkpoint is corrupted?", "chosen": "Use the latest intact checkpoint or rerun from a clean save point.", "rejected": "Keep loading the same corrupted file until it works."},
    {"kind": "debugging", "user": "What should I inspect if gradients are always zero?", "chosen": "Check whether loss depends on the parameters and whether backward ran.", "rejected": "Increase the batch size because zero gradients mean too little data."},
    {"kind": "debugging", "user": "What should I do if token IDs exceed the model vocabulary?", "chosen": "Use the tokenizer that matches the model artifact.", "rejected": "Clamp large token IDs to the final vocabulary entry."},
    {"kind": "debugging", "user": "What should I do if the notebook uses too much memory?", "chosen": "Close other kernels and reduce batch size or context length.", "rejected": "Open another notebook so the run has more space."},
    {"kind": "concise_style", "user": "Explain a tensor in one sentence.", "chosen": "A tensor is an array of numbers with a shape.", "rejected": "A tensor is a thing that is very important and can be many things in many ways."},
    {"kind": "concise_style", "user": "Explain gradient descent in one sentence.", "chosen": "Gradient descent updates parameters in the direction that lowers loss.", "rejected": "Gradient descent is when the model goes around and learns stuff until it gets better somehow."},
    {"kind": "concise_style", "user": "Explain a tokenizer in one sentence.", "chosen": "A tokenizer converts text into token IDs a model can process.", "rejected": "A tokenizer is a big complicated text thing that does all sorts of text processing."},
    {"kind": "concise_style", "user": "Explain logits in one sentence.", "chosen": "Logits are unnormalized scores before softmax turns them into probabilities.", "rejected": "Logits are numbers and then there is softmax and then it is kind of probability-like."},
    {"kind": "concise_style", "user": "Explain attention in one sentence.", "chosen": "Attention lets each token mix information from relevant earlier tokens.", "rejected": "Attention is when the model pays attention and does transformer things."},
    {"kind": "concise_style", "user": "Explain an embedding in one sentence.", "chosen": "An embedding is a learned vector representation of a token.", "rejected": "An embedding is a number thing that stores meaning in a mysterious way."},
    {"kind": "concise_style", "user": "Explain DPO in one sentence.", "chosen": "DPO trains a policy to prefer chosen responses over rejected ones while staying near a reference.", "rejected": "DPO is a preference thing that makes answers better in some way."},
    {"kind": "concise_style", "user": "Explain RAG in one sentence.", "chosen": "RAG retrieves relevant context and includes it in the prompt before generation.", "rejected": "RAG is when the model gets documents and is smarter."},
    {"kind": "concise_style", "user": "Explain a learning rate in one sentence.", "chosen": "A learning rate controls how large each parameter update is.", "rejected": "A learning rate is a setting you change when training feels wrong."},
    {"kind": "concise_style", "user": "Explain a validation set in one sentence.", "chosen": "A validation set estimates performance on data not used for training updates.", "rejected": "A validation set is extra training data you look at sometimes."},
    {"kind": "concise_style", "user": "Explain a tool call in one sentence.", "chosen": "A tool call is structured text the runtime parses and executes.", "rejected": "A tool call is when the model magically uses an external program."},
    {"kind": "concise_style", "user": "Explain a system prompt in one sentence.", "chosen": "A system prompt gives hidden instructions that frame the assistant behavior.", "rejected": "A system prompt is just the user's message with a different name."},
    {"kind": "grounding", "user": "Context: Ada wrote the report on Monday. Who wrote the report?", "chosen": "Ada wrote the report.", "rejected": "Ben wrote the report."},
    {"kind": "grounding", "user": "Context: The meeting starts at 3 PM. When does the meeting start?", "chosen": "It starts at 3 PM.", "rejected": "It starts at noon."},
    {"kind": "grounding", "user": "Context: The blue key opens the lab. Which key opens the lab?", "chosen": "The blue key opens the lab.", "rejected": "The red key opens the lab."},
    {"kind": "grounding", "user": "Context: Module 14 covers DPO. Which module covers DPO?", "chosen": "Module 14 covers DPO.", "rejected": "Module 13 covers DPO."},
    {"kind": "grounding", "user": "Context: The file is named notes.md. What is the filename?", "chosen": "notes.md", "rejected": "summary.txt"},
    {"kind": "grounding", "user": "Context: The dataset has 120 examples. How many examples are in the dataset?", "chosen": "120 examples.", "rejected": "100 examples."},
    {"kind": "grounding", "user": "Context: The model ran on MPS. What device did it use?", "chosen": "It used MPS.", "rejected": "It used CUDA."},
    {"kind": "grounding", "user": "Context: The validation loss rose after step 800. What rose after step 800?", "chosen": "The validation loss rose.", "rejected": "The training batch size rose."},
    {"kind": "grounding", "user": "Context: The answer must be JSON only. What format is required?", "chosen": "JSON only.", "rejected": "Plain English paragraphs."},
    {"kind": "grounding", "user": "Context: The checkpoint path is artifacts/models/BaseLM-SFT. What is the checkpoint path?", "chosen": "artifacts/models/BaseLM-SFT", "rejected": "data/cache/BaseLM"},
    {"kind": "tool_json", "user": "Call calculator for 23 * 17.", "chosen": "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"23*17\"}}", "rejected": "calculator 23 times 17 please"},
    {"kind": "tool_json", "user": "Call calculator for 10 + 5.", "chosen": "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"10+5\"}}", "rejected": "{\"tool\":\"calculator\",\"expr\":\"10+5\"}"},
    {"kind": "tool_json", "user": "Call search for gradient clipping.", "chosen": "{\"name\":\"search\",\"arguments\":{\"query\":\"gradient clipping\"}}", "rejected": "Search for gradient clipping."},
    {"kind": "tool_json", "user": "Call read_file for docs/syllabus.md.", "chosen": "{\"name\":\"read_file\",\"arguments\":{\"path\":\"docs/syllabus.md\"}}", "rejected": "{\"name\":\"read_file\",\"path\":\"docs/syllabus.md\"}"},
    {"kind": "tool_json", "user": "Call list_files for docs/modules.", "chosen": "{\"name\":\"list_files\",\"arguments\":{\"path\":\"docs/modules\"}}", "rejected": "{name:list_files,args:docs/modules}"},
    {"kind": "tool_json", "user": "Call calculator for 7 squared.", "chosen": "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"7*7\"}}", "rejected": "The answer is probably 49."},
    {"kind": "tool_json", "user": "Call lookup_weather for Boston.", "chosen": "{\"name\":\"lookup_weather\",\"arguments\":{\"location\":\"Boston\"}}", "rejected": "{\"name\":\"lookup_weather\",\"location\":\"Boston\"}"},
    {"kind": "tool_json", "user": "Call python_eval for len('abc').", "chosen": "{\"name\":\"python_eval\",\"arguments\":{\"code\":\"len('abc')\"}}", "rejected": "{\"name\":\"python_eval\",\"arguments\":\"len('abc')\"}"},
    {"kind": "tool_json", "user": "Call summarize_file for README.md.", "chosen": "{\"name\":\"summarize_file\",\"arguments\":{\"path\":\"README.md\"}}", "rejected": "I will summarize README.md now."},
    {"kind": "tool_json", "user": "Call grep for TODO in g2c.", "chosen": "{\"name\":\"grep\",\"arguments\":{\"pattern\":\"TODO\",\"path\":\"g2c\"}}", "rejected": "{\"grep\":\"TODO\",\"where\":\"g2c\"}"},
    {"kind": "actual_question", "user": "Name one primary color and nothing else.", "chosen": "Red.", "rejected": "Primary colors are important in art."},
    {"kind": "actual_question", "user": "What command runs the tests?", "chosen": "pytest", "rejected": "Testing is useful because it catches bugs."},
    {"kind": "actual_question", "user": "Which section comes after The big idea?", "chosen": "Concepts to internalize.", "rejected": "The big idea explains the module."},
    {"kind": "actual_question", "user": "Answer the final word of this sentence: models learn from data.", "chosen": "data.", "rejected": "Models can learn many patterns."},
    {"kind": "actual_question", "user": "Give the file extension of notebook.ipynb.", "chosen": ".ipynb", "rejected": "A notebook is an interactive document."},
    {"kind": "actual_question", "user": "What does SFT stand for?", "chosen": "Supervised fine-tuning.", "rejected": "SFT is used after pretraining."},
    {"kind": "actual_question", "user": "What is the requested output format: JSON or markdown?", "chosen": "JSON.", "rejected": "Both JSON and markdown are common formats."},
    {"kind": "actual_question", "user": "How many items are in this list: A, B, C?", "chosen": "Three.", "rejected": "The list contains letters."},
    {"kind": "actual_question", "user": "What is the first token in '<|user|>'?", "chosen": "<|user|>", "rejected": "It is a chat marker."},
    {"kind": "actual_question", "user": "Should the answer include extra explanation? Answer yes or no.", "chosen": "No.", "rejected": "Extra explanation can sometimes help."},
]

print(f"preference rows: {len(preference_rows)}")
print(json.dumps(preference_rows[0], indent=2))
assert len(preference_rows) >= 100

Save your preference rows if you want the dataset as a reusable artifact. The notebook keeps this disabled by default so exploratory edits do not overwrite a hand-curated file.

In [ ]:
SAVE_PREFERENCE_JSON = False
PREFERENCE_JSON_PATH = repo_root / "data" / "dpo" / "preferences.json"

if SAVE_PREFERENCE_JSON:
    PREFERENCE_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    PREFERENCE_JSON_PATH.write_text(
        json.dumps(preference_rows, indent=2) + "\n",
        encoding="utf-8",
    )
    print("saved", PREFERENCE_JSON_PATH.relative_to(repo_root))
else:
    print("not saved; set SAVE_PREFERENCE_JSON = True when the dataset is ready")

Check token lengths before training. If chosen completions are systematically longer, DPO can learn length instead of preference.

In [ ]:
def response_ids(text: str) -> list[int]:
    return tokenizer.encode_with_vocab_size(text + template.END, policy_model.vocab_size)

length_rows = []
for row in preference_rows:
    chosen_len = len(response_ids(row["chosen"]))
    rejected_len = len(response_ids(row["rejected"]))
    ratio = max(chosen_len, rejected_len) / max(1, min(chosen_len, rejected_len))
    length_rows.append((row["kind"], chosen_len, rejected_len, ratio, row["user"]))

print(f"{'kind':<12} {'chosen':>6} {'rejected':>8} {'ratio':>6}  prompt")
for kind, chosen_len, rejected_len, ratio, user in length_rows[:20]:
    print(f"{kind:<12} {chosen_len:>6} {rejected_len:>8} {ratio:>6.2f}  {user[:54]}")

avg_chosen = sum(row[1] for row in length_rows) / len(length_rows)
avg_rejected = sum(row[2] for row in length_rows) / len(length_rows)
print("\naverage chosen tokens:", round(avg_chosen, 2))
print("average rejected tokens:", round(avg_rejected, 2))
print("max length ratio:", round(max(row[3] for row in length_rows), 2))

## Encode preference triples

A DPO prompt is the user turn plus the assistant role marker. The chosen and rejected responses are completions after that marker, and both include `<|end|>` so the model can learn which answer should stop.

In [ ]:
def render_dpo_prompt(user_text: str) -> str:
    return template.render([{"role": "user", "content": user_text}]) + f"{template.ASSISTANT}\n"


def encode_preference(row: dict) -> PreferenceExample:
    prompt_ids = tokenizer.encode_with_vocab_size(
        render_dpo_prompt(row["user"]),
        policy_model.vocab_size,
    )
    chosen_ids = tokenizer.encode_with_vocab_size(
        row["chosen"] + template.END,
        policy_model.vocab_size,
    )
    rejected_ids = tokenizer.encode_with_vocab_size(
        row["rejected"] + template.END,
        policy_model.vocab_size,
    )
    return PreferenceExample(
        prompt_ids=prompt_ids,
        chosen_ids=chosen_ids,
        rejected_ids=rejected_ids,
    )

encoded_rows = [(row, encode_preference(row)) for row in preference_rows]
encoded_preferences = [ex for _, ex in encoded_rows]

first_row, first_ex = encoded_rows[0]
print("prompt text:")
print(render_dpo_prompt(first_row["user"]))
print("prompt ids:", len(first_ex.prompt_ids))
print("chosen ids:", len(first_ex.chosen_ids), tokenizer.decode(first_ex.chosen_ids))
print("rejected ids:", len(first_ex.rejected_ids), tokenizer.decode(first_ex.rejected_ids))
assert max(max(ex.prompt_ids + ex.chosen_ids + ex.rejected_ids) for ex in encoded_preferences) < policy_model.vocab_size

Inspect the collator on two examples. The mask should be `1` only on response targets, never on the prompt or padding.

In [ ]:
DPO_MAX_SEQ_LEN = min(128, policy_model.max_seq_len)

cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
    encoded_preferences[:2],
    max_seq_len=DPO_MAX_SEQ_LEN,
    pad_id=pad_id,
)
print("chosen x/y/mask:", cx.shape, cy.shape, cm.shape)
print("rejected x/y/mask:", rx.shape, ry.shape, rm.shape)
print("chosen mask sums:", cm.sum(dim=1).tolist())
print("rejected mask sums:", rm.sum(dim=1).tolist())


def masked_target_text(y: torch.Tensor, mask: torch.Tensor) -> str:
    ids = [int(token_id) for token_id, keep in zip(y.tolist(), mask.tolist()) if int(keep) == 1]
    return tokenizer.decode(ids)

print("\nchosen target text:", masked_target_text(cy[0], cm[0]))
print("rejected target text:", masked_target_text(ry[0], rm[0]))

## Train/validation split

In [ ]:
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(encoded_rows), generator=generator).tolist()
split = max(1, int(0.8 * len(perm)))
train_indices = perm[:split]
val_indices = perm[split:] or perm[-1:]

train_rows = [encoded_rows[i][0] for i in train_indices]
val_rows = [encoded_rows[i][0] for i in val_indices]
train_examples = [encoded_rows[i][1] for i in train_indices]
val_examples = [encoded_rows[i][1] for i in val_indices]

print("train examples:", len(train_examples))
print("val examples:", len(val_examples))

def average(values):
    return sum(values) / len(values) if values else 0.0


def percent(numerator: int, denominator: int) -> float:
    return 100.0 * numerator / denominator if denominator else 0.0


chosen_lengths = [len(ex.chosen_ids) for ex in encoded_preferences]
rejected_lengths = [len(ex.rejected_ids) for ex in encoded_preferences]
chosen_longer = sum(c > r for c, r in zip(chosen_lengths, rejected_lengths))
train_kinds = [encoded_rows[i][0]["kind"] for i in train_indices]
val_kinds = [encoded_rows[i][0]["kind"] for i in val_indices]
all_kinds = sorted({row["kind"] for row in preference_rows})

print("\npreference dataset summary")
print(f"{'metric':<30} value")
print("-" * 44)
print(f"{'pairs':<30} {len(encoded_preferences):>6}")
print(f"{'train pairs':<30} {len(train_examples):>6}")
print(f"{'validation pairs':<30} {len(val_examples):>6}")
print(f"{'avg chosen response tokens':<30} {average(chosen_lengths):>6.1f}")
print(f"{'avg rejected response tokens':<30} {average(rejected_lengths):>6.1f}")
print(f"{'chosen longer':<30} {percent(chosen_longer, len(encoded_preferences)):>5.1f}%")

print("\nkind distribution")
print(f"{'kind':<18} {'total':>5} {'train':>5} {'val':>5}")
print("-" * 36)
for kind in all_kinds:
    total = sum(row["kind"] == kind for row in preference_rows)
    train = sum(item == kind for item in train_kinds)
    val = sum(item == kind for item in val_kinds)
    print(f"{kind:<18} {total:>5} {train:>5} {val:>5}")

## Exercise 2 - Step-0 DPO sanity check

Before training, the policy and reference are identical copies. The DPO margin is therefore zero and the loss should be `log(2) ≈ 0.6931`. This is the deepest single sanity check for your data path.

In [ ]:
def preference_batch_metrics(policy, reference, examples, *, beta: float, max_seq_len: int):
    cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
        examples,
        max_seq_len=max_seq_len,
        pad_id=pad_id,
    )
    device = model_device(policy)
    cx, cy, cm = cx.to(device), cy.to(device), cm.to(device)
    rx, ry, rm = rx.to(device), ry.to(device), rm.to(device)
    with torch.no_grad():
        policy_c = sequence_logprob(policy(cx), cy, cm)
        policy_r = sequence_logprob(policy(rx), ry, rm)
        ref_c = sequence_logprob(reference(cx), cy, cm)
        ref_r = sequence_logprob(reference(rx), ry, rm)
        loss, metrics = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=beta)
    return {
        "loss": loss.item(),
        "chosen_reward": metrics["chosen_reward"].item(),
        "rejected_reward": metrics["rejected_reward"].item(),
        "reward_margin": metrics["reward_margin"].item(),
        "accuracy": metrics["accuracy"].item(),
    }

initial_metrics = preference_batch_metrics(
    policy_model,
    ref_model,
    train_examples[: min(8, len(train_examples))],
    beta=0.1,
    max_seq_len=DPO_MAX_SEQ_LEN,
)
print(initial_metrics)
print("log(2):", math.log(2))
assert abs(initial_metrics["loss"] - math.log(2)) < 1e-3

## Exercise 3 - Train DPO

The defaults are intentionally conservative. DPO is about 3x the wall-clock of SFT at the same model size because it runs policy/reference on chosen/rejected sequences. Increase steps only after the loss, reward margin, and samples make sense.

In [ ]:
IS_HF_ARTIFACT = policy_artifact.manifest.get("kind") == "huggingface_causal_lm"

DPO_CONFIG = {
    "max_seq_len": DPO_MAX_SEQ_LEN,
    "pad_id": pad_id,
    "beta": 0.1,
    "batch_size": 1 if IS_HF_ARTIFACT else 4,
    "max_steps": 120 if IS_HF_ARTIFACT else 300,
    "max_lr": 5e-5 if IS_HF_ARTIFACT else 1e-4,
    "min_lr": 5e-6 if IS_HF_ARTIFACT else 1e-5,
    "warmup_steps": 10,
    "weight_decay": 0.0,
    "grad_clip": 1.0,
    "eval_every": 20 if IS_HF_ARTIFACT else 50,
    "eval_iters": 3,
    "log_every": 5 if IS_HF_ARTIFACT else 10,
    "device": TRAIN_DEVICE,
}
DPO_CONFIG

In [ ]:
trainer = DPOTrainer(
    policy_model,
    ref_model=ref_model,
    examples=train_examples,
    generator=torch.Generator().manual_seed(SEED),
    **DPO_CONFIG,
)

history = train_dpo_with_progress(
    f"{policy_artifact.name} DPO",
    trainer,
    eval_examples=val_examples,
)
plot_dpo_history(history)

## Exercise 4 - Compare SFT vs DPO behavior

Use the same prompts, seed, and sampling settings. The reference model is your original SFT checkpoint; the policy model is now DPO-tuned.

In [ ]:
COMPARISON_PROMPTS = [
    "What is the capital of Spain?",
    "What is the capital of France?",
    "Explain logits in one sentence.",
    "If you are unsure about an answer, what should you say?",
    "Return exactly one color.",
]


def compare_models(prompts: list[str], *, seed: int = SEED) -> None:
    for prompt in prompts:
        print("=" * 72)
        print("user:", prompt)
        show_response("SFT/reference", ref_model, prompt, max_new_tokens=80, seed=seed)
        show_response("DPO/policy", policy_model, prompt, max_new_tokens=80, seed=seed)

compare_models(COMPARISON_PROMPTS)

Score held-out preference pairs directly. Positive reward margin means the DPO policy favors the chosen completion more than the frozen SFT reference does.

In [ ]:
def score_preference_example(policy, reference, ex: PreferenceExample, *, beta: float = 0.1) -> dict[str, float]:
    metrics = preference_batch_metrics(policy, reference, [ex], beta=beta, max_seq_len=DPO_MAX_SEQ_LEN)
    return metrics

print(f"{'kind':<12} {'margin':>8} {'acc':>5}  prompt")
for row, ex in zip(val_rows, val_examples):
    metrics = score_preference_example(policy_model, ref_model, ex, beta=DPO_CONFIG["beta"])
    print(f"{row['kind']:<12} {metrics['reward_margin']:>8.3f} {metrics['accuracy']:>5.2f}  {row['user'][:52]}")

### Written reflection

Question: Where did DPO clearly improve the SFT model, and where did it fail or make behavior worse?

Answer: 

## Exercise 5 - Optional beta sweep

Run this only after the baseline run works. The quickest useful sweep is three betas over fewer steps. The goal is not a perfect model; it is to see low beta drift, mid beta learning, and high beta under-movement.

Trains three fresh policies, so expect a few minutes. Skip the cell if you do not want to wait.

In [ ]:
BETA_VALUES = [0.05, 0.1, 0.3]
BETA_SWEEP_STEPS = 80

beta_results = {}
for beta in BETA_VALUES:
    fresh_policy = load_model_artifact_with_tokenizer(
        SFT_ARTIFACT_NAME,
        repo_root=repo_root,
        device=TRAIN_DEVICE,
    ).model
    fresh_ref = load_model_artifact_with_tokenizer(
        SFT_ARTIFACT_NAME,
        repo_root=repo_root,
        device=TRAIN_DEVICE,
    ).model
    config = {**DPO_CONFIG, "beta": beta, "max_steps": BETA_SWEEP_STEPS}
    sweep_trainer = DPOTrainer(
        fresh_policy,
        ref_model=fresh_ref,
        examples=train_examples,
        generator=torch.Generator().manual_seed(SEED),
        **config,
    )
    sweep_history = train_dpo_with_progress(
        f"beta={beta}",
        sweep_trainer,
        eval_examples=val_examples,
    )
    beta_results[beta] = {
        "history": sweep_history,
        "model": fresh_policy,
    }

for beta, result in beta_results.items():
    h = result["history"]
    print(
        beta,
        "final loss", round(h["train_loss"][-1], 4),
        "final margin", round(h["reward_margin"][-1], 4),
        "final acc", round(h["accuracy"][-1], 3),
    )

### Beta sweep notes

Question: Which beta moved the model enough without visibly damaging its base behavior?

Answer: 

## Exercise 6 - Save the DPO artifact

This preserves the original SFT artifact and writes a separate DPO artifact for Module 15 evaluation. If your samples got worse, set `SAVE_DPO_ARTIFACT = False`, tune the run, and save only when the artifact is worth reusing.

In [ ]:
DPO_ARTIFACT_NAME = (
    SFT_ARTIFACT_NAME[:-4] + "-DPO"
    if SFT_ARTIFACT_NAME.endswith("-SFT")
    else f"{SFT_ARTIFACT_NAME}-DPO"
)
SAVE_DPO_ARTIFACT = True

if SAVE_DPO_ARTIFACT:
    training_config = {
        **DPO_CONFIG,
        "sft_artifact": SFT_ARTIFACT_NAME,
        "num_examples": len(encoded_preferences),
        "num_train_examples": len(train_examples),
        "num_val_examples": len(val_examples),
        "preference_rows": len(preference_rows),
    }
    if policy_artifact.manifest.get("kind") == "huggingface_causal_lm":
        artifact_dir = save_huggingface_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            tokenizer=tokenizer,
            base_artifact_name=SFT_ARTIFACT_NAME,
            training_config=training_config,
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    else:
        model_config = dict(policy_artifact.manifest["model_config"])
        artifact_dir = save_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            model_config=model_config,
            training_config=training_config,
            tokenizer_artifact_name=policy_artifact.manifest["tokenizer_artifact"],
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            seed=SEED,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    print("saved", artifact_dir.relative_to(repo_root))
else:
    print("not saved")

## Deliverable notes

Question: Explain why the initial DPO loss is `log(2)`.

Answer: 

Question: Explain why the reference model must stay frozen.

Answer: 

Question: What is one preference-dataset bias you checked for?

Answer: 